# Python Finalization: A Second Advanced Tutorial
## 16 **new** problems with step-by-step solutions

**Focus:** `__del__`, incomplete construction, object ownership, hidden references, cyclic GC diagnostics, weak finalization, and dependable cleanup.

**Format:** scenario → prediction → tiny experiment → interpretation → next experiment → complete solution → assertions → takeaways. This deliberately mirrors the supplied lecture notebook's explanation-rich, incremental teaching style, rather than providing one big solution per problem.

**Requirements:** Python **3.11+**, standard library only. Run cells **top to bottom** in a fresh kernel. No network, external packages, real sockets, or persistent files are required. One experiment launches a local Python subprocess to avoid altering your notebook's GC debug state.

**Terminology correction:** `del name` removes a binding; it does *not* call `__del__` directly. Finalizer timing is not a resource-management contract. CPython often runs an object's finalizer as its last reference disappears, while other implementations may run it later. `__del__` exceptions are unraisable to the caller. Always use explicit `close()` / `with` / `async with` for required cleanup.

### Where these examples come from

**From the supplied tutorial:** the central questions about `del`, extra references, exception tracebacks, unraisable exceptions, and why context managers are preferable to finalizers.

**New advanced extensions in this notebook:** failed constructors, cooperative inheritance, cleanup reentrancy, generator frames, cancellation, method caches, slotted objects, `gc.DEBUG_SAVEALL`, finalizer cancellation, ownership transfer, combined failures, copying, module-teardown simulation, and an integrated lifecycle design. These extensions are additional Python material—not claims that they appeared in the supplied lecture.

In [1]:
import asyncio
import copy
import gc
import json
import pickle
import subprocess
import sys
import weakref
from contextlib import ExitStack, closing
from dataclasses import dataclass
from functools import lru_cache

print(f"Python {sys.version_info.major}.{sys.version_info.minor} — tutorial ready")
assert sys.version_info >= (3, 11), "Python 3.11+ is required."

Python 3.13 — tutorial ready


### Laboratory protocol

1. Keep a plain `events` list to record effects, rather than relying on the order of notebook print output.
2. Use `weakref.ref` to inspect existence without extending lifetime. Do not save the referent in a variable while testing its death.
3. `gc.collect()` is a diagnostic aid, **not** a promise that all external references disappear.
4. Tests of prompt reclamation are labeled CPython-oriented; the portable lesson is to close explicitly.
5. Run each problem's assertion cell before moving on. The final cell checks all recorded passes.

In [2]:
PASSED = []

def passed(number, explanation):
    PASSED.append(number)
    print(f"PASS {number:02d}: {explanation}")

## Problem 01 — A constructor raises before its fields exist

**Scenario.** An initializer allocates an instance, then raises at different points.

**Your challenge.** Write a finalizer that does not assume `__init__` completed; explain why cleanup still must not depend on it.

Try to predict the result of each small experiment before running it. The solution is developed in stages rather than revealed in one large code block.

### Step 1 — predict the failure mode

Python may finalize an instance whose `__init__` raised. A finalizer such as `self.resource.close()` can then raise `AttributeError` because `resource` was never assigned. The caller cannot catch that finalizer error by catching the constructor error.

**Prediction:** Is it safe to assume an attribute set *later* in `__init__` always exists inside `__del__`? No. Observe the early and late failure paths without creating a real resource.

In [3]:
def experiment_01():
    events = []

    class Fragile:
        def __init__(self, where):
            self.where = where
            if where == "early":
                raise RuntimeError("failed before resource assignment")
            self.resource = "acquired"
            raise RuntimeError("failed after resource assignment")

        def __del__(self):
            # Diagnostic only: no required resource cleanup here.
            events.append((getattr(self, "where", "unknown"),
                           getattr(self, "resource", "missing")))

    for point in ("early", "late"):
        try:
            Fragile(point)
        except RuntimeError as exc:
            print(type(exc).__name__, str(exc))
    gc.collect()
    return events

seen_01 = experiment_01()
print("Observed finalizer state:", seen_01)

RuntimeError failed before resource assignment
RuntimeError failed after resource assignment
Observed finalizer state: [('early', 'missing'), ('late', 'acquired')]


### Step 2 — interpret the result

The **early** instance lacks `resource`; the **late** instance has it. Even if the output appears promptly in CPython, neither output timing nor completeness of initialization is guaranteed across implementations.

A *defensive* finalizer may use `getattr(self, "resource", None)`, but that does not turn finalizers into reliable cleanup. Make the constructor responsible for rolling back partially acquired resources instead.

### Step 3 — solve the real acquisition problem

Use an `ExitStack` locally inside a factory-style operation. If a later acquisition raises, the stack exits already-acquired resources before the error reaches the caller. Transfer ownership out of the stack **only after** all acquisitions succeed. We will implement this more fully in Problem 16.

### Verification — make the conclusion executable

These assertions check the intended behavior. Where collection timing is implementation-dependent, the code explicitly requests collection or tests deterministic cleanup rather than assuming a precise finalizer timestamp.

In [4]:
assert sorted(seen_01) == [("early", "missing"), ("late", "acquired")]
passed(1, "partial initialization does not justify field assumptions")

PASS 01: partial initialization does not justify field assumptions


### What this teaches

`__del__` may see incomplete state. Favor transactional acquisition and explicit rollback rather than trying to repair a failed constructor from a finalizer.

## Problem 02 — Cooperative inheritance: the missing base cleanup

**Scenario.** A subclass defines `__del__` and unintentionally overrides its parent finalizer.

**Your challenge.** Trace which cleanup hooks run, then make the order explicit with `super()`.

Try to predict the result of each small experiment before running it. The solution is developed in stages rather than revealed in one large code block.

### Step 1 — build a minimal inheritance chain

Special methods are ordinary methods with respect to overriding: Python does not automatically call a parent `__del__`. For this exercise, the hooks only append to a list, so no real resource is at stake.

In [5]:
def experiment_02():
    events = []

    class Parent:
        def __del__(self):
            events.append("parent")

    class ForgetfulChild(Parent):
        def __del__(self):
            events.append("forgetful-child")

    obj = ForgetfulChild()
    del obj
    gc.collect()
    first = events[:]

    class CooperativeChild(Parent):
        def __del__(self):
            try:
                events.append("cooperative-child")
            finally:
                super().__del__()

    second = CooperativeChild()
    del second
    gc.collect()
    return first, events

first_02, all_02 = experiment_02()
print("Without super:", first_02)
print("With super:", all_02[len(first_02):])

Without super: ['forgetful-child']
With super: ['cooperative-child', 'parent']


### Step 2 — explain the contrast

The forgetful child hides its parent's method. The cooperative child explicitly calls `super().__del__()` from `finally`, so the base hook runs even if the preceding child cleanup raises. This is an illustrative inheritance technique, **not** a recommendation to coordinate essential cleanup through a chain of finalizers.

In multiple inheritance, every participating class needs a compatible cooperative design; an unrelated mixin can break the chain. A well-defined public `close()` protocol is easier to call and test.

### Verification — make the conclusion executable

These assertions check the intended behavior. Where collection timing is implementation-dependent, the code explicitly requests collection or tests deterministic cleanup rather than assuming a precise finalizer timestamp.

In [6]:
assert first_02 == ["forgetful-child"]
assert all_02 == ["forgetful-child", "cooperative-child", "parent"]
passed(2, "subclass finalizers must chain explicitly, if used at all")

PASS 02: subclass finalizers must chain explicitly, if used at all


### What this teaches

Overriding `__del__` replaces the inherited hook; the base method is not automatic. Reserve required lifecycle ordering for explicit APIs.

## Problem 03 — A cleanup callback calls `close()` again

**Scenario.** A close operation notifies an observer that immediately calls close on the same object.

**Your challenge.** Prevent recursive cleanup and demonstrate exactly-once resource release.

Try to predict the result of each small experiment before running it. The solution is developed in stages rather than revealed in one large code block.

### Step 1 — see the reentrancy bug before coding

Consider `close()` that calls an observer first and marks `closed=True` afterward. If the observer calls `close()` again, the second call also sees `closed=False`; recursive close can repeat indefinitely or double-release a resource.

**Key design rule:** mark the object closed *before* invoking arbitrary user callbacks. This is an explicit-close problem that `__del__` makes even harder to debug.

In [7]:
class ReentrantLease03:
    def __init__(self, events):
        self.events = events
        self.closed = False
        self.on_close = None

    def close(self):
        if self.closed:
            self.events.append("already-closed")
            return
        self.closed = True  # State transition before external code.
        self.events.append("release")
        if self.on_close is not None:
            self.on_close()

events_03 = []
lease_03 = ReentrantLease03(events_03)
lease_03.on_close = lease_03.close
lease_03.close()
print(events_03)

['release', 'already-closed']


### Step 2 — derive the invariant

The nested call observes the already-closed state. The trace includes a second *attempt*, but exactly one `release`. In production, you might avoid recording the no-op at all.

Note that `lease.on_close = lease.close` creates a bound-method/self cycle. We made this safe **because cleanup is explicit**, not because we expect the cycle's eventual finalization to run promptly.

In [8]:
lease_03.close()
print("After a third close attempt:", events_03)
# Break the illustrative self-cycle; none of the assertions depend on GC timing.
lease_03.on_close = None

After a third close attempt: ['release', 'already-closed', 'already-closed']


### Verification — make the conclusion executable

These assertions check the intended behavior. Where collection timing is implementation-dependent, the code explicitly requests collection or tests deterministic cleanup rather than assuming a precise finalizer timestamp.

In [9]:
assert events_03.count("release") == 1
assert events_03.count("already-closed") == 2
passed(3, "reentrant close preserves exactly-once release")

PASS 03: reentrant close preserves exactly-once release


### What this teaches

An idempotent close method should establish its terminal state before executing callbacks. Do not depend on finalizers to enforce this invariant.

## Problem 04 — A paused generator is the hidden owner

**Scenario.** A generator pauses at `yield`, leaving a local variable in its suspended frame.

**Your challenge.** Prove that dropping an apparent object name is not enough, then close the generator.

Try to predict the result of each small experiment before running it. The solution is developed in stages rather than revealed in one large code block.

### Step 1 — construct an object that you can observe weakly

An active or suspended generator frame can keep local objects alive. This is different from the lecture's exception-traceback example, but follows the same principle: an unexpected frame is an unexpected owner.

In [10]:
def experiment_04():
    events = []

    class Tracked:
        def __del__(self):
            events.append("finalized")

    def suspended():
        item = Tracked()
        yield weakref.ref(item)  # Do not yield item itself.

    iterator = suspended()
    probe = next(iterator)
    gc.collect()
    alive_while_paused = probe() is not None
    iterator.close()  # Abandons the frame, including its local 'item'.
    gc.collect()
    alive_after_close = probe() is not None
    return alive_while_paused, alive_after_close, events

paused_04, after_04, events_04 = experiment_04()
print("While paused:", paused_04, "| after close:", after_04)
print("Diagnostic:", events_04)

While paused: True | after close: False
Diagnostic: ['finalized']


### Step 2 — add one extra twist

If the generator had yielded the *object itself*, the consumer's variable would hold a strong reference even after `iterator.close()`. A weak probe avoids that measurement error. In CPython, the object will normally disappear immediately after the suspended frame releases it; here we also request collection before checking.

### Verification — make the conclusion executable

These assertions check the intended behavior. Where collection timing is implementation-dependent, the code explicitly requests collection or tests deterministic cleanup rather than assuming a precise finalizer timestamp.

In [11]:
assert paused_04 is True
assert after_04 is False
assert events_04 == ["finalized"]
passed(4, "a suspended generator frame can keep locals alive")

PASS 04: a suspended generator frame can keep locals alive


### What this teaches

Manage generators that own resources with explicit `close()` or appropriate context management. Merely dropping a name outside a suspended generator does not clear its locals.

## Problem 05 — Cancellation must release an async resource

**Scenario.** An asynchronous task owns a resource and is cancelled during an `await`.

**Your challenge.** Guarantee cleanup by placing it in `finally`, cancel the task, then await its completion.

Try to predict the result of each small experiment before running it. The solution is developed in stages rather than revealed in one large code block.

### Step 1 — analyze the wrong instinct

A synchronous `__del__` is **not** an async cleanup hook: it cannot reliably await a coroutine or coordinate task completion. Even if a resource eventually becomes unreachable, cancellation might occur long before its finalizer runs.

**Prediction:** What must the caller do after `task.cancel()` to learn when cleanup has completed? Await the cancelled task and handle `CancelledError`.

In [12]:
async def experiment_05():
    events = []
    ready = asyncio.Event()

    class AsyncOwned:
        def close(self):
            events.append("closed")

    async def worker():
        owned = AsyncOwned()
        try:
            ready.set()
            await asyncio.Event().wait()  # Wait until cancelled.
        finally:
            owned.close()

    task = asyncio.create_task(worker())
    await ready.wait()
    task.cancel()
    try:
        await task
    except asyncio.CancelledError:
        events.append("cancel-observed")
    return events

events_05 = await experiment_05()
print(events_05)

['closed', 'cancel-observed']


### Step 2 — explain the ordering

`finally` executes as cancellation unwinds the coroutine. Only after awaiting the task does the caller know that synchronous cleanup in that `finally` has run. For an *asynchronous* close operation, use `async with` / `__aexit__` and design cancellation handling explicitly; a destructor still cannot await it.

Jupyter kernels normally have a running event loop, so this notebook uses top-level `await`. In a standalone Python script, use `asyncio.run(experiment_05())` instead.

### Verification — make the conclusion executable

These assertions check the intended behavior. Where collection timing is implementation-dependent, the code explicitly requests collection or tests deterministic cleanup rather than assuming a precise finalizer timestamp.

In [13]:
assert events_05 == ["closed", "cancel-observed"]
passed(5, "awaited cancellation confirms deterministic cleanup")

PASS 05: awaited cancellation confirms deterministic cleanup


### What this teaches

Cancellation is a structured-control-flow event. Finish cleanup inside `finally` or an async context manager; never schedule required async cleanup from `__del__`.

## Problem 06 — `lru_cache` retains `self` as a cache key

**Scenario.** A decorated instance method seems to be a harmless speed optimization.

**Your challenge.** Locate the strong reference keeping instances alive and redesign the cache boundary.

Try to predict the result of each small experiment before running it. The solution is developed in stages rather than revealed in one large code block.

### Step 1 — understand the reference graph

An `lru_cache` memoizes calls using their arguments. On an instance method, `self` is one of those arguments, so the cache may own the instance strongly. The cache wrapper lives on the class even after a local name is deleted.

In [14]:
def experiment_06():
    class Calculation:
        @lru_cache(maxsize=8)
        def square(self, number):
            return number * number

    obj = Calculation()
    probe = weakref.ref(obj)
    assert obj.square(7) == 49
    del obj
    gc.collect()
    cached_owner_alive = probe() is not None
    info_before = Calculation.square.cache_info()
    Calculation.square.cache_clear()
    gc.collect()
    still_alive = probe() is not None
    return cached_owner_alive, still_alive, info_before

cached_06, after_clear_06, info_06 = experiment_06()
print("Retained by cache:", cached_06)
print("After cache_clear:", after_clear_06)
print("Cache stats before clearing:", info_06)

Retained by cache: True
After cache_clear: False
Cache stats before clearing: CacheInfo(hits=0, misses=1, maxsize=8, currsize=1)


### Step 2 — redesign: cache by immutable value, not by object identity

If a calculation only depends on an immutable parameter, move the cached computation to a module-level helper or `@staticmethod`. Then the cache stores the parameter, not the instance.

The following helper intentionally accepts only integers; it never receives a `Calculation` instance.

In [15]:
@lru_cache(maxsize=8)
def square_by_value(number: int) -> int:
    return number * number

assert square_by_value(7) == 49
assert square_by_value(7) == 49
print("Value-only cache:", square_by_value.cache_info())
square_by_value.cache_clear()

Value-only cache: CacheInfo(hits=1, misses=1, maxsize=8, currsize=1)


### Verification — make the conclusion executable

These assertions check the intended behavior. Where collection timing is implementation-dependent, the code explicitly requests collection or tests deterministic cleanup rather than assuming a precise finalizer timestamp.

In [16]:
assert cached_06 is True
assert after_clear_06 is False
assert info_06.currsize == 1
passed(6, "cached bound-method arguments can be strong owners")

PASS 06: cached bound-method arguments can be strong owners


### What this teaches

When diagnosing a delayed `__del__`, inspect caches as well as ordinary variables. Cache immutable keys rather than live resource-owning objects when possible.

## Problem 07 — Why a slotted object cannot be weakly referenced

**Scenario.** You want to attach a weak finalizer to a compact slotted dataclass.

**Your challenge.** Reproduce the `TypeError`, add weak-reference support, and ensure the callback does not capture the referent.

Try to predict the result of each small experiment before running it. The solution is developed in stages rather than revealed in one large code block.

### Step 1 — test a compact class

A class with `__slots__` generally needs a `__weakref__` slot to support weak references. Python 3.11 introduced `weakref_slot=True` for `@dataclass(slots=True)`.

In [17]:
@dataclass(slots=True)
class Compact07:
    token: str

compact_07 = Compact07("x")
try:
    weakref.ref(compact_07)
except TypeError as exc:
    print("Expected:", str(exc))
    weakref_unavailable_07 = True
else:
    weakref_unavailable_07 = False

Expected: cannot create weak reference to 'Compact07' object


### Step 2 — add the missing slot and register a safe callback

The callback below receives an immutable token and a separate log. It does **not** close over `obj`, store `obj` as an argument, or use a bound method of `obj`—all of which could accidentally keep the referent alive.

In [18]:
@dataclass(slots=True, weakref_slot=True)
class WeakReady07:
    token: str

log_07 = []
obj_07 = WeakReady07("resource-7")
probe_07 = weakref.ref(obj_07)
finalizer_07 = weakref.finalize(obj_07, log_07.append, obj_07.token)
del obj_07
gc.collect()
print("Referent available:", probe_07() is not None)
print("Fallback callback:", log_07)

Referent available: False
Fallback callback: ['resource-7']


### Verification — make the conclusion executable

These assertions check the intended behavior. Where collection timing is implementation-dependent, the code explicitly requests collection or tests deterministic cleanup rather than assuming a precise finalizer timestamp.

In [19]:
assert weakref_unavailable_07 is True
assert probe_07() is None
assert log_07 == ["resource-7"]
assert finalizer_07.alive is False
passed(7, "slotted dataclass opts into weak references explicitly")

PASS 07: slotted dataclass opts into weak references explicitly


### What this teaches

Weak-reference support is a class-layout decision. A weak finalizer callback must not accidentally hold the target through its function or arguments.

## Problem 08 — `gc.DEBUG_SAVEALL` does not mean “uncollectable”

**Scenario.** A cyclic object appears in `gc.garbage` during a debugging session.

**Your challenge.** Explain the difference between debug-mode retention and true garbage, without polluting the notebook interpreter.

Try to predict the result of each small experiment before running it. The solution is developed in stages rather than revealed in one large code block.

### Step 1 — isolate an invasive debugging experiment

`gc.set_debug(gc.DEBUG_SAVEALL)` deliberately appends otherwise-unreachable objects to `gc.garbage` instead of freeing them. Seeing a finalizable object there **does not** prove it is inherently uncollectable. Modern Python supports finalization of many cycles involving `__del__`.

Because GC debug flags affect the entire process, run the experiment in a short-lived subprocess.

In [20]:
script_08 = r"""
import gc
import json

log = []
class Node:
    def __del__(self):
        log.append("finalized")

node = Node()
node.link = node
node = None
gc.set_debug(gc.DEBUG_SAVEALL)
gc.collect()
print(json.dumps({
    "node_in_garbage": sum(type(x) is Node for x in gc.garbage),
    "finalizer_calls": log.count("finalized")
}))
"""
result_08 = subprocess.run(
    [sys.executable, "-c", script_08],
    capture_output=True, text=True, check=True,
)
data_08 = json.loads(result_08.stdout.strip())
print(data_08)

{'node_in_garbage': 1, 'finalizer_calls': 1}


### Step 2 — interpret precisely

The node can be both **finalized** and **present in `gc.garbage`**: debug mode itself retained the unreachable object. Outside debug mode, cyclic GC can normally clear this kind of simple self-cycle. Some objects can still be uncollectable for special reasons (for example, certain extension types); do not infer a universal guarantee.

Using a subprocess keeps your own notebook's `gc` flags and `gc.garbage` untouched.

### Verification — make the conclusion executable

These assertions check the intended behavior. Where collection timing is implementation-dependent, the code explicitly requests collection or tests deterministic cleanup rather than assuming a precise finalizer timestamp.

In [21]:
assert data_08["node_in_garbage"] == 1
assert data_08["finalizer_calls"] == 1
passed(8, "DEBUG_SAVEALL intentionally retains unreachable objects")

PASS 08: DEBUG_SAVEALL intentionally retains unreachable objects


### What this teaches

Diagnose collection with a controlled experiment. `gc.garbage` under `DEBUG_SAVEALL` reports deliberate debug retention, not necessarily an intrinsic leak.

## Problem 09 — Cancel and transfer a `weakref.finalize` fallback

**Scenario.** You registered an emergency finalizer, but later handed ownership to a new manager.

**Your challenge.** Use `detach()` to cancel the old registration; compare it with calling the finalizer manually.

Try to predict the result of each small experiment before running it. The solution is developed in stages rather than revealed in one large code block.

### Step 1 — learn the two different operations

A `weakref.finalize` object can be invoked explicitly: this runs its callback at most once. In contrast, `detach()` marks it dead **without running it**, and returns a tuple containing the original referent and callback details. That returned tuple temporarily owns the referent—do not keep it around while testing reclamation.

In [22]:
events_09 = []

class Target09:
    pass

first_09 = Target09()
probe_09 = weakref.ref(first_09)
fin_09 = weakref.finalize(first_09, events_09.append, "old-owner-release")
transfer_record_09 = fin_09.detach()
print("Registration still live?", fin_09.alive)
print("Transferred callback args:", transfer_record_09[2])
del transfer_record_09
del first_09
gc.collect()
print("Target alive:", probe_09() is not None, "| callbacks:", events_09)

Registration still live? False
Transferred callback args: ('old-owner-release',)
Target alive: False | callbacks: []


### Step 2 — compare explicit invocation

Explicit `finalizer()` runs the fallback immediately and disarms it. It can be useful for a *best-effort fallback*, but is not a substitute for a design with clear `close()` ownership. The `atexit` flag only controls whether a still-live finalizer is eligible to run at interpreter exit.

In [23]:
second_09 = Target09()
fin2_09 = weakref.finalize(second_09, events_09.append, "manual-release")
fin2_09.atexit = False
fin2_09()  # Invoke now; the callback runs exactly once.
fin2_09()  # No second invocation.
del second_09
gc.collect()
print(events_09)

['manual-release']


### Verification — make the conclusion executable

These assertions check the intended behavior. Where collection timing is implementation-dependent, the code explicitly requests collection or tests deterministic cleanup rather than assuming a precise finalizer timestamp.

In [24]:
assert fin_09.alive is False
assert probe_09() is None
assert events_09 == ["manual-release"]
assert fin2_09.alive is False
passed(9, "detach cancels; call executes and disarms")

PASS 09: detach cancels; call executes and disarms


### What this teaches

Use explicit ownership handoff: disarm obsolete fallback callbacks on transfer, and do not accidentally extend a referent's lifetime by storing the tuple returned by `detach()`.

## Problem 10 — Adapt a legacy object with `contextlib.closing`

**Scenario.** An external API exposes `close()` but has no `__enter__` or `__exit__`.

**Your challenge.** Write a `with` block that closes the object even when the body raises, without modifying its class.

Try to predict the result of each small experiment before running it. The solution is developed in stages rather than revealed in one large code block.

### Step 1 — recognize an ownership boundary

`closing(resource)` adapts an object with a `close()` method into a context manager. It calls `close()` on exit but does not suppress body exceptions. This is usually preferable to adding `__del__` to an old API.

In [25]:
class LegacyStream10:
    def __init__(self, log):
        self.log = log
        self.closed = False

    def read(self):
        if self.closed:
            raise RuntimeError("already closed")
        return "payload"

    def close(self):
        if not self.closed:
            self.closed = True
            self.log.append("close")

log_10 = []
stream_10 = LegacyStream10(log_10)
try:
    with closing(stream_10) as source:
        print("Read:", source.read())
        raise ValueError("body failed")
except ValueError as exc:
    caught_10 = str(exc)
print("After the exception:", stream_10.closed, log_10)

Read: payload
After the exception: True ['close']


### Step 2 — challenge the design

What if a caller already owns this `stream` and merely lends it to your function? Wrapping it in `closing` would incorrectly close someone else's resource. Use `closing` only when your scope has accepted responsibility for cleanup. **Ownership is a contract, not a property of the class.**

### Verification — make the conclusion executable

These assertions check the intended behavior. Where collection timing is implementation-dependent, the code explicitly requests collection or tests deterministic cleanup rather than assuming a precise finalizer timestamp.

In [26]:
assert caught_10 == "body failed"
assert stream_10.closed
assert log_10 == ["close"]
passed(10, "closing adapts close-only resources with exception-safe cleanup")

PASS 10: closing adapts close-only resources with exception-safe cleanup


### What this teaches

Use `contextlib.closing` at a scope that actually owns the resource. The `with` statement guarantees the exit path in normal execution and during exceptions.

## Problem 11 — Transfer a whole cleanup stack with `pop_all()`

**Scenario.** A setup phase acquires several resources, but another component must outlive the setup scope.

**Your challenge.** Move cleanup responsibility exactly once instead of closing resources when the setup scope exits.

Try to predict the result of each small experiment before running it. The solution is developed in stages rather than revealed in one large code block.

### Step 1 — register callbacks in acquisition order

`ExitStack` unwinds callbacks in last-in, first-out order. `pop_all()` *moves* registered callbacks into a new stack without executing them. This is a specific ownership-transfer operation, distinct from the previous notebook's rollback-on-failure exercise.

In [27]:
log_11 = []
with ExitStack() as acquisition_11:
    acquisition_11.callback(log_11.append, "release-A")
    acquisition_11.callback(log_11.append, "release-B")
    owner_11 = acquisition_11.pop_all()
    print("During setup, after transfer:", log_11)
print("After setup's own with block:", log_11)
owner_11.close()
print("After new owner closes:", log_11)

During setup, after transfer: []
After setup's own with block: []
After new owner closes: ['release-B', 'release-A']


### Step 2 — verify ownership did not duplicate

Closing the *original* stack after `pop_all()` must do nothing; closing the *new* stack runs callbacks once, in reverse order. Calling `close()` on the already-empty new stack again must also do nothing.

In [28]:
owner_11.close()
acquisition_11.close()
print("After repeated close attempts:", log_11)

After repeated close attempts: ['release-B', 'release-A']


### Verification — make the conclusion executable

These assertions check the intended behavior. Where collection timing is implementation-dependent, the code explicitly requests collection or tests deterministic cleanup rather than assuming a precise finalizer timestamp.

In [29]:
assert log_11 == ["release-B", "release-A"]
passed(11, "pop_all transfers cleanup ownership without execution")

PASS 11: pop_all transfers cleanup ownership without execution


### What this teaches

A cleanup stack can model ownership transfer directly. Acquire locally, transfer only on success, then close from the new owner.

## Problem 12 — Both the body and cleanup fail: preserve both errors

**Scenario.** A context-manager body raises `ValueError`, then its cleanup raises `OSError`.

**Your challenge.** Expose both failures without silently losing the original problem.

Try to predict the result of each small experiment before running it. The solution is developed in stages rather than revealed in one large code block.

### Step 1 — understand the edge case

A naïve `__exit__` that calls a failing `close()` raises the cleanup exception while the body exception is active. The body exception may still be available through exception chaining, but callers tend to notice only the new top-level error. In Python 3.11+, an `ExceptionGroup` can deliberately expose both independent failures.

In [30]:
class DualFailure12:
    def __enter__(self):
        return self

    def close(self):
        raise OSError("close failed")

    def __exit__(self, exc_type, exc, tb):
        try:
            self.close()
        except OSError as cleanup_error:
            if exc is not None:
                raise ExceptionGroup(
                    "body and cleanup both failed", [exc, cleanup_error]
                ) from None
            raise
        return False

try:
    with DualFailure12():
        raise ValueError("work failed")
except ExceptionGroup as group_12:
    observed_12 = [(type(e).__name__, str(e)) for e in group_12.exceptions]
print(observed_12)

[('ValueError', 'work failed'), ('OSError', 'close failed')]


### Step 2 — test the one-error branch separately

If the body succeeds but cleanup fails, there is only one meaningful exception to raise; an `ExceptionGroup` is unnecessary. Never make `__exit__` return `True` merely to hide the body's failure.

In [31]:
try:
    with DualFailure12():
        pass
except OSError as exc:
    cleanup_only_12 = str(exc)
print("Cleanup-only failure:", cleanup_only_12)

Cleanup-only failure: close failed


### Verification — make the conclusion executable

These assertions check the intended behavior. Where collection timing is implementation-dependent, the code explicitly requests collection or tests deterministic cleanup rather than assuming a precise finalizer timestamp.

In [32]:
assert observed_12 == [
    ("ValueError", "work failed"),
    ("OSError", "close failed"),
]
assert cleanup_only_12 == "close failed"
passed(12, "body and cleanup errors are both observable")

PASS 12: body and cleanup errors are both observable


### What this teaches

A context manager should document exception precedence. `ExceptionGroup` is one explicit option when both work and cleanup fail; do not silently suppress either error.

## Problem 13 — A weak proxy is not a safe permanent handle

**Scenario.** A weak proxy behaves like an object until its referent disappears.

**Your challenge.** Handle `ReferenceError` and contrast proxy access with a `weakref.ref` existence check.

Try to predict the result of each small experiment before running it. The solution is developed in stages rather than revealed in one large code block.

### Step 1 — observe successful proxy dispatch

`weakref.proxy(obj)` does not keep `obj` alive. While the referent exists, `proxy.some_attribute` forwards access to it. Once the referent dies, most proxy operations raise `ReferenceError`. A weak proxy is not a resource owner.

In [33]:
class Payload13:
    def __init__(self, value):
        self.value = value

owner_13 = Payload13(123)
proxy_13 = weakref.proxy(owner_13)
probe_13 = weakref.ref(owner_13)
print("Live proxy value:", proxy_13.value)
del owner_13
gc.collect()
print("Weak reference result:", probe_13())

Live proxy value: 123
Weak reference result: None


### Step 2 — handle the dead referent correctly

A check like `if proxy:` is not a safe liveness test—it may itself raise. Instead, request a strong reference by calling the `weakref.ref`; if you get an object, retain that local reference for the duration of the operation.

In [34]:
try:
    _ = proxy_13.value
except ReferenceError:
    dead_proxy_caught_13 = True
else:
    dead_proxy_caught_13 = False

strong_13 = probe_13()
if strong_13 is None:
    print("Referent gone; skip operation")
else:
    print("Safe while strong_13 is in scope:", strong_13.value)

Referent gone; skip operation


### Verification — make the conclusion executable

These assertions check the intended behavior. Where collection timing is implementation-dependent, the code explicitly requests collection or tests deterministic cleanup rather than assuming a precise finalizer timestamp.

In [35]:
assert dead_proxy_caught_13 is True
assert probe_13() is None
passed(13, "weak proxies fail after referent collection")

PASS 13: weak proxies fail after referent collection


### What this teaches

A weak proxy allows convenient access, not guaranteed availability. Promote `weakref.ref()` to a temporary strong local reference before performing a multi-step operation.

## Problem 14 — Copying a resource owner can cause double cleanup

**Scenario.** A class owns a non-shareable external handle, yet callers try `copy`, `deepcopy`, and pickle.

**Your challenge.** Enforce a single-owner contract and offer an immutable snapshot instead of duplicating the handle.

Try to predict the result of each small experiment before running it. The solution is developed in stages rather than revealed in one large code block.

### Step 1 — diagnose the ownership bug

An ordinary shallow copy can duplicate an object's Python attributes without duplicating the operating-system handle they represent. If each copy later runs a finalizer or `close()`, the underlying handle may be released twice. Deepcopy and pickling may be inappropriate for the same reason.

For a strictly unique owner, reject those operations explicitly rather than hoping finalizers resolve ownership.

In [36]:
class UniqueOwner14:
    def __init__(self, log, token):
        self.log = log
        self.token = token
        self.closed = False

    def __copy__(self):
        raise TypeError("resource ownership cannot be copied")

    def __deepcopy__(self, memo):
        raise TypeError("resource ownership cannot be deep-copied")

    def __reduce_ex__(self, protocol):
        raise TypeError("a live resource owner cannot be pickled")

    def snapshot(self):
        return (self.token, self.closed)  # Shareable immutable data only.

    def close(self):
        if not self.closed:
            self.closed = True
            self.log.append(self.token)

log_14 = []
owner_14 = UniqueOwner14(log_14, "token-14")
print("Snapshot:", owner_14.snapshot())

Snapshot: ('token-14', False)


### Step 2 — test three prohibited operations independently

We catch each `TypeError` because rejection is the intended result, not a notebook failure. The snapshot remains available without copying the resource itself.

In [37]:
rejections_14 = []
for name, operation in [
    ("shallow", lambda: copy.copy(owner_14)),
    ("deep", lambda: copy.deepcopy(owner_14)),
    ("pickle", lambda: pickle.dumps(owner_14)),
]:
    try:
        operation()
    except TypeError:
        rejections_14.append(name)
owner_14.close()
owner_14.close()
print("Rejected:", rejections_14, "| releases:", log_14)

Rejected: ['shallow', 'deep', 'pickle'] | releases: ['token-14']


### Verification — make the conclusion executable

These assertions check the intended behavior. Where collection timing is implementation-dependent, the code explicitly requests collection or tests deterministic cleanup rather than assuming a precise finalizer timestamp.

In [38]:
assert rejections_14 == ["shallow", "deep", "pickle"]
assert log_14 == ["token-14"]
assert owner_14.snapshot() == ("token-14", True)
passed(14, "non-copyable ownership prevents accidental handle duplication")

PASS 14: non-copyable ownership prevents accidental handle duplication


### What this teaches

Define whether objects can be copied or serialized. A resource owner is not automatically ordinary data; an immutable snapshot can carry metadata without carrying ownership.

## Problem 15 — A module dependency disappears before `__del__` runs

**Scenario.** A finalizer looks up a module-level logging function that is no longer present.

**Your challenge.** Simulate teardown safely, show the failure, and explain why capturing a callback is only a defensive measure.

Try to predict the result of each small experiment before running it. The solution is developed in stages rather than revealed in one large code block.

### Step 1 — specify the shutdown hazard

At interpreter shutdown, module globals may already have been replaced or torn down when a finalizer runs. The exact order is not guaranteed. To make a **deterministic demonstration**, we simulate this loss in a subprocess by setting a dependency to `None` before deleting two objects.

One finalizer looks up the global dynamically; the other saved its callback when the object was created. Neither should perform critical cleanup during real shutdown.

In [39]:
script_15 = r"""
import gc
import json

events = []
emit = lambda item, sink=events: sink.append(item)

class LateLookup:
    def __del__(self):
        if emit is None:
            events.append("global-missing")
        else:
            emit("dynamic-success")

class Captured:
    def __init__(self, callback):
        self.callback = callback
    def __del__(self):
        self.callback("captured-success")

late = LateLookup()
safe = Captured(emit)
emit = None  # Simulated teardown, not actual interpreter shutdown.
del late, safe
gc.collect()
print(json.dumps(events))
"""
result_15 = subprocess.run(
    [sys.executable, "-c", script_15],
    capture_output=True, text=True, check=True,
)
events_15 = json.loads(result_15.stdout.strip())
print(events_15)

['global-missing', 'captured-success']


### Step 2 — draw the narrow conclusion

The captured function survives **this controlled dependency removal**, while dynamic lookup sees `None`. That does not prove captured callbacks will always work during real interpreter shutdown: they may rely on other already-removed state, and finalizers are not guaranteed to run for objects still alive on exit.

For essential shutdown work, define an explicit application lifecycle and call its `close()` methods while dependencies are known to exist.

### Verification — make the conclusion executable

These assertions check the intended behavior. Where collection timing is implementation-dependent, the code explicitly requests collection or tests deterministic cleanup rather than assuming a precise finalizer timestamp.

In [40]:
assert events_15 == ["global-missing", "captured-success"]
passed(15, "shutdown-like dependency loss defeats dynamic global lookup")

PASS 15: shutdown-like dependency loss defeats dynamic global lookup


### What this teaches

Capture needed dependencies as a defensive technique, but guarantee important cleanup before interpreter shutdown rather than in `__del__`.

## Problem 16 — Capstone: transactional acquisition with borrowed and owned objects

**Scenario.** A service assembles two managed components and optionally borrows a third.

**Your challenge.** Guarantee rollback on partial setup, reverse-order release, exactly-once close, and no cleanup of the borrowed object.

Try to predict the result of each small experiment before running it. The solution is developed in stages rather than revealed in one large code block.

### Step 1 — write the ownership contract first

Our service must satisfy five invariants:

- Owned components enter in acquisition order and exit in reverse order.
- A failure halfway through acquisition closes everything already acquired.
- Borrowed components are *not* entered into our cleanup stack.
- `close()` can be called repeatedly without releasing anything twice.
- Exceptions from the service's body propagate after cleanup.

None of these can safely be delegated to finalizer timing. We need explicit, local ownership.

In [41]:
class Component16:
    def __init__(self, name, log):
        self.name = name
        self.log = log

    def __enter__(self):
        self.log.append("enter:" + self.name)
        return self

    def __exit__(self, exc_type, exc, tb):
        self.log.append("exit:" + self.name)
        return False

### Step 2 — implement the transactional acquisition boundary

A *temporary* `ExitStack` owns the newly acquired components until setup succeeds. `pop_all()` transfers those callbacks to the long-lived service. If a later acquisition raises, the temporary stack rolls back automatically. The borrowed component is only stored as data; we do not enter or close it.

In [42]:
class ServiceBundle16:
    def __init__(self, log, borrowed=None, fail_second=False):
        self.log = log
        self.borrowed = borrowed
        self.fail_second = fail_second
        self.components = []
        self._owned = ExitStack()
        self._opened = False

    def __enter__(self):
        if self._opened:
            raise RuntimeError("bundle is already open")
        with ExitStack() as provisional:
            first = provisional.enter_context(Component16("A", self.log))
            if self.fail_second:
                raise ValueError("could not acquire B")
            second = provisional.enter_context(Component16("B", self.log))
            transferred = provisional.pop_all()
        self.components = [first, second]
        self._owned = transferred
        self._opened = True
        return self

    def close(self):
        if self._opened:
            self._opened = False
            self._owned.close()

    def __exit__(self, exc_type, exc, tb):
        self.close()
        return False

### Step 3 — success: observe acquisition, transfer, and teardown

The caller owns the bundle via `with`. A borrowed component is not automatically ours to close, so its log remains untouched.

In [43]:
owned_log_16 = []
borrowed_log_16 = []
borrowed_16 = Component16("BORROWED", borrowed_log_16)
bundle_16 = ServiceBundle16(owned_log_16, borrowed=borrowed_16)
with bundle_16 as active:
    assert [c.name for c in active.components] == ["A", "B"]
    assert active.borrowed is borrowed_16
    print("While open:", owned_log_16)
bundle_16.close()  # Must be a no-op.
print("After exit:", owned_log_16)
print("Borrowed component untouched:", borrowed_log_16)

While open: ['enter:A', 'enter:B']
After exit: ['enter:A', 'enter:B', 'exit:B', 'exit:A']
Borrowed component untouched: []


### Step 4 — failure: roll back the first acquisition

`__enter__` raises before ownership transfers to the bundle. The temporary stack executes A's exit callback on its way out, and the caller sees the original `ValueError`. There is no need to rely on either `__del__` or an `__exit__` method for a context manager that never entered successfully.

In [44]:
failed_log_16 = []
broken_16 = ServiceBundle16(failed_log_16, fail_second=True)
try:
    with broken_16:
        raise AssertionError("body must never start")
except ValueError as exc:
    failure_16 = str(exc)
broken_16.close()  # No transferred ownership exists.
print("Failure:", failure_16, "| rollback:", failed_log_16)

Failure: could not acquire B | rollback: ['enter:A', 'exit:A']


### Step 5 — body failure: unwind and propagate

After a successful enter, the body may fail for unrelated reasons. The bundle's `__exit__` closes owned resources and returns `False`, preserving the original exception.

In [45]:
error_log_16 = []
try:
    with ServiceBundle16(error_log_16):
        raise LookupError("body exploded")
except LookupError as exc:
    body_error_16 = str(exc)
print("Body exception:", body_error_16, "| cleanup:", error_log_16)

Body exception: body exploded | cleanup: ['enter:A', 'enter:B', 'exit:B', 'exit:A']


### Verification — make the conclusion executable

These assertions check the intended behavior. Where collection timing is implementation-dependent, the code explicitly requests collection or tests deterministic cleanup rather than assuming a precise finalizer timestamp.

In [46]:
assert owned_log_16 == ["enter:A", "enter:B", "exit:B", "exit:A"]
assert borrowed_log_16 == []
assert failed_log_16 == ["enter:A", "exit:A"]
assert failure_16 == "could not acquire B"
assert error_log_16 == ["enter:A", "enter:B", "exit:B", "exit:A"]
assert body_error_16 == "body exploded"
passed(16, "ownership, rollback, LIFO release, and error propagation all hold")

PASS 16: ownership, rollback, LIFO release, and error propagation all hold


### What this teaches

The durable design is explicit, testable ownership. Use a provisional cleanup stack for setup, transfer ownership only after success, and never close borrowed components.

---
# Integrated review: explain, do not just memorize

Before revealing the short answers, reason through these prompts:

**A.** A class has `__del__`; its `__init__` can raise. What should `__del__` assume about instance fields?

**B.** Why can a paused generator or cached instance method postpone finalization after `del obj`?

**C.** Why might `gc.garbage` contain a finalizable object during debugging?

**D.** Distinguish `weakref.finalize(...).detach()` from calling the finalizer.

**E.** A setup function borrows a file but owns a socket. Which object belongs in its cleanup stack?

**F.** The body and cleanup both fail. What information must a robust error policy preserve?

## Worked review answers

**A.** Nothing beyond fields checked defensively with `getattr` or similar; required rollback belongs in the construction/acquisition path.

**B.** Suspended frames and method caches can own strong references even after one visible binding is removed.

**C.** `gc.DEBUG_SAVEALL` explicitly retains unreachable objects for inspection. Their presence does not, by itself, establish intrinsic uncollectability.

**D.** `detach()` disarms without invoking and returns a tuple that includes the referent; calling executes the callback once and disarms.

**E.** Only the socket your setup owns. The borrowed file's original owner retains cleanup responsibility.

**F.** Preserve visibility of both failures where relevant (for example, through an explicit `ExceptionGroup` policy), rather than silently discarding the original body exception.

In [47]:
expected = list(range(1, 17))
assert PASSED == expected, f"Missing/out-of-order checks: {PASSED!r}"
print(f"ALL {len(PASSED)} NEW TUTORIAL PROBLEMS PASSED")

ALL 16 NEW TUTORIAL PROBLEMS PASSED


## Compact reference: choose the right lifecycle mechanism

| Situation | Prefer | Avoid |
|---|---|---|
| Required synchronous cleanup | `with`, `close()`, `finally` | Depending on `__del__` timing |
| Required asynchronous cleanup | `async with`, awaited shutdown | Trying to await from `__del__` |
| Partially acquired resources | Temporary `ExitStack` | Cleanup solely in failed `__init__` |
| Transfer several owned resources | `ExitStack.pop_all()` | Duplicating cleanup callbacks |
| Legacy close-only object you own | `contextlib.closing` | Closing a borrowed object |
| Best-effort GC fallback | Carefully designed `weakref.finalize` | Callbacks retaining the target |
| Probe whether an object is alive | `weakref.ref` | Holding an extra strong reference |
| Object with unique external handle | Non-copyable owner + immutable snapshot | Blind `copy` / `deepcopy` / pickle |

**Final principle:** A finalizer helps explain object lifetime; a documented ownership protocol makes correct cleanup testable. This is also why the original lecture favors context managers for real resources.